# Workshop Setup

One-click provisioning for the SQLBits 2026 workshop **"Adapting to Microsoft Fabric: A SQL Server Practitioner's Way Forward"**.

## The easiest way to set it up

1. Create a workspace in Microsoft Fabric.
2. Create a new notebook in that workspace.
3. Copy the code cell below into the new notebook.
4. Run all cells.

## Another way

1. Create a workspace in Microsoft Fabric.
2. Download this notebook from GitHub.
3. Import the notebook into your workspace.
4. Run all cells.

What happens when you run it:

1. Two Lakehouses (`bronze`, `silver`) and one Warehouse (`gold`) are created in the current workspace. Existing items with the same name are reused, so the notebook is safe to re-run.
2. The workshop notebooks are pulled from GitHub into your workspace, each pre-bound to the default lakehouse (and any additional data sources) configured in `NOTEBOOKS_WITH_DEFAULT_LAKEHOUSES`.
3. The `DataBuilder` notebook is executed so the `customers` and `orders` source tables (plus the parquet + CSV file copies) are ready before the workshop starts.


In [ ]:
# Install workshop dependencies. %pip may restart the kernel, so keep this in
# its own first cell so no other state is lost when the kernel resets.
# sempy_labs is the supported Python path for Fabric Warehouse provisioning.
# notebookutils has lakehouse.create but no warehouse.create.
# Pin PyJWT>=2.6.0 alongside to satisfy fsspec-wrapper's transitive requirement.
%pip install "semantic-link-labs" "PyJWT>=2.6.0" --quiet


In [ ]:
# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------

LAKEHOUSE_NAMES = [
    {"Name": "bronze", "ENABLE_LAKEHOUSE_SCHEMAS": 1}, 
    {"Name": "silver", "ENABLE_LAKEHOUSE_SCHEMAS": 1}]

WAREHOUSE_NAMES = ["gold"]

GITHUB_ACCOUNT = "https://raw.githubusercontent.com/ChristianHenrikReich/fabric_sqlserver_practitioners_fullday_handson/refs/heads/main"

# No external data files to seed, DataBuilder generates everything at runtime.
DATA_FILES = []

# (github path, data source spec):
#   default_lakehouse        = lakehouse that the notebook is attached to as default
#   additional_data_sources  = extra lakehouses / warehouses the notebook can read/write
NOTEBOOKS_WITH_DEFAULT_LAKEHOUSES = [
    (f"{GITHUB_ACCOUNT}/notebooks/module_0_1_DataBuilder.ipynb",          {"default_lakehouse": "silver", "additional_data_sources": ["bronze", "gold"]}),
    (f"{GITHUB_ACCOUNT}/notebooks/module_1_1_TimeTravel.ipynb",           {"default_lakehouse": "silver", "additional_data_sources": []}),
    (f"{GITHUB_ACCOUNT}/notebooks/module_1_2_Optimize.ipynb",             {"default_lakehouse": "silver", "additional_data_sources": []}),
    (f"{GITHUB_ACCOUNT}/notebooks/module_2_1_Language.ipynb",             {"default_lakehouse": "silver", "additional_data_sources": []}),
    (f"{GITHUB_ACCOUNT}/notebooks/module_2_2_Catalyst.ipynb",             {"default_lakehouse": "silver", "additional_data_sources": []}),
    (f"{GITHUB_ACCOUNT}/notebooks/module_2_3_Optimizing.ipynb",           {"default_lakehouse": "silver", "additional_data_sources": []}),
    (f"{GITHUB_ACCOUNT}/notebooks/module_2_4_Merge.ipynb",                {"default_lakehouse": "silver", "additional_data_sources": []}),
    (f"{GITHUB_ACCOUNT}/notebooks/module_2_5_Native_Execution_Engine.ipynb", {"default_lakehouse": "silver", "additional_data_sources": []}),
    (f"{GITHUB_ACCOUNT}/notebooks/module_3_1_restore_point.ipynb",        {"default_lakehouse": "silver", "additional_data_sources": ["gold"]}),
    (f"{GITHUB_ACCOUNT}/notebooks/module_3_2_Interoperability.ipynb",     {"default_lakehouse": "silver", "additional_data_sources": ["gold"]}),
]

MAX_WORKERS = 5

# ---------------------------------------------------------------------------
from concurrent.futures import ThreadPoolExecutor
import json as _json
import os
import time
import requests
import notebookutils

def _item_id(item):
    """Extract the item id from a notebookutils info object or a plain dict."""
    if item is None:
        return None
    props = getattr(item, "properties", None)
    if isinstance(props, dict):
        for k in ("id", "Id", "workspaceId"):
            if k == "id" and k in props:
                return props[k]
            if k == "Id" and k in props:
                return props[k]
    if isinstance(item, dict):
        return item.get("id") or item.get("Id")
    return getattr(item, "id", None)


def _create_schema_enabled_lakehouse(name):
    """Create a schema-enabled lakehouse via the Fabric REST API."""
    import sempy.fabric as fabric
    client = fabric.FabricRestClient()
    workspace_id = fabric.get_workspace_id()
    client.post(
        f"v1/workspaces/{workspace_id}/lakehouses",
        json={
            "displayName": name,
            "description": f"Workshop lakehouse: {name}",
            "creationPayload": {"enableSchemas": True},
        },
    )
    # Long-running operation; poll notebookutils.get until the item is visible.
    for _ in range(30):
        try:
            return notebookutils.lakehouse.get(name=name)
        except Exception:
            time.sleep(2)
    return notebookutils.lakehouse.get(name=name)


def _create_or_get_lakehouses(lakehouse_specs):
    """Create or get each lakehouse. Each spec is a dict:
        {"Name": "<name>", "ENABLE_LAKEHOUSE_SCHEMAS": 0|1}
    The schema flag is per-lakehouse. Warehouses ignore this entirely.
    """
    print("Get or create Lakehouses")
    lakehouses = {}
    for spec in lakehouse_specs:
        name = spec["Name"]
        enable_schemas = bool(spec.get("ENABLE_LAKEHOUSE_SCHEMAS", 0))
        try:
            info = notebookutils.lakehouse.get(name=name)
            print(f"  Found Lakehouse: {name}")
        except Exception:
            if enable_schemas:
                info = _create_schema_enabled_lakehouse(name)
                print(f"  Created schema-enabled Lakehouse: {name}")
            else:
                info = notebookutils.lakehouse.create(name)
                print(f"  Created Lakehouse: {name}")
        lakehouses[name] = info
    return lakehouses


def _create_or_get_warehouses(warehouse_names):
    if not warehouse_names:
        print("No Warehouses to create")
        return {}
    from sempy_labs.warehouse import create_warehouse, list_warehouses
    print("Get or create Warehouses")
    df = list_warehouses()
    existing = {row["Warehouse Name"]: row for _, row in df.iterrows()}
    for name in warehouse_names:
        if name in existing:
            print(f"  Found Warehouse: {name}")
        else:
            create_warehouse(warehouse=name, description=f"Workshop warehouse: {name}")
            print(f"  Created Warehouse: {name}")
    # Re-list so newly created warehouses appear with ids.
    df = list_warehouses()
    return {row["Warehouse Name"]: row for _, row in df.iterrows() if row["Warehouse Name"] in warehouse_names}


def _get_from_git(url):
    print(f"  Fetching {url}")
    r = requests.get(url)
    r.raise_for_status()
    return r


def _save_file_from_git(url, save_path):
    content = _get_from_git(url).content
    print(f"  Writing {save_path}")
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    with open(save_path, "wb") as f:
        f.write(content)


def _inject_data_sources(content_text, spec, lakehouse_ids, warehouse_ids, workspace_id):
    """Rewrite the notebook content so its metadata points at the default lakehouse
    plus any additional lakehouses/warehouses listed in the spec."""
    nb = _json.loads(content_text)
    meta = nb.setdefault("metadata", {})
    # Start from a clean dependencies block so attendees see only what Setup configured.
    meta["dependencies"] = {}
    deps = meta["dependencies"]

    default_name = spec.get("default_lakehouse")
    additional   = spec.get("additional_data_sources") or []

    known_lh = []
    if default_name and default_name in lakehouse_ids:
        default_id = lakehouse_ids[default_name]
        known_lh.append({"id": default_id})
        deps["lakehouse"] = {
            "default_lakehouse": default_id,
            "default_lakehouse_name": default_name,
            "default_lakehouse_workspace_id": workspace_id,
            "known_lakehouses": known_lh,
        }

    known_wh = []
    for n in additional:
        if n in lakehouse_ids and not any(k["id"] == lakehouse_ids[n] for k in known_lh):
            known_lh.append({"id": lakehouse_ids[n]})
        elif n in warehouse_ids:
            known_wh.append({"id": warehouse_ids[n], "type": "Datawarehouse"})

    if known_wh:
        deps["warehouse"] = {"known_warehouses": known_wh}

    return _json.dumps(nb, indent=1)


def _save_notebook_from_git(url, spec, lakehouse_ids, warehouse_ids, workspace_id):
    content = _get_from_git(url).text
    content = _inject_data_sources(content, spec, lakehouse_ids, warehouse_ids, workspace_id)
    name = os.path.splitext(os.path.basename(url))[0]
    default_lh = spec.get("default_lakehouse") or ""
    try:
        notebookutils.notebook.create(name=name, content=content, defaultLakehouse=default_lh)
        print(f"  Imported notebook: {name} (default: {default_lh or 'none'})")
    except Exception:
        notebookutils.notebook.updateDefinition(name=name, content=content, defaultLakehouse=default_lh)
        print(f"  Updated notebook:  {name} (default: {default_lh or 'none'})")


# ---------------------------------------------------------------------------
# 1. Create / get the medallion lakehouses and the gold warehouse
# ---------------------------------------------------------------------------
lakehouses = _create_or_get_lakehouses(LAKEHOUSE_NAMES)
warehouses = _create_or_get_warehouses(WAREHOUSE_NAMES)

# Resolve ids for metadata injection.
import sempy.fabric as _fabric
WORKSPACE_ID = _fabric.get_workspace_id()
LAKEHOUSE_IDS = {n: _item_id(info) for n, info in lakehouses.items()}
WAREHOUSE_IDS = {}
for n, row in warehouses.items():
    # sempy_labs.list_warehouses columns include "Warehouse Name" and "Warehouse Id" (or similar)
    wid = row.get("Warehouse Id") if hasattr(row, "get") else None
    if wid is None:
        for k in ("Id", "id", "WarehouseId"):
            if k in row:
                wid = row[k]
                break
    WAREHOUSE_IDS[n] = wid

# ---------------------------------------------------------------------------
# 2. (Optional) mount a lakehouse and seed data files from GitHub
# ---------------------------------------------------------------------------
if DATA_FILES:
    primary = LAKEHOUSE_NAMES[0]["Name"]
    mount_name = primary.lower()
    print(f"Mounting {primary} to /{mount_name}")
    notebookutils.fs.mount(
        lakehouses[primary].properties["abfsPath"],
        f"/{mount_name}",
        {"fileCacheTimeout": 0, "timeout": 120},
    )
    mount_path = notebookutils.fs.getMountPath(f"/{mount_name}")
    for url, dest in DATA_FILES:
        _save_file_from_git(url, f"{mount_path}{dest}")

# ---------------------------------------------------------------------------
# 3. Import workshop notebooks into the workspace (parallel)
# ---------------------------------------------------------------------------
print("Importing workshop notebooks")
def _do_import(t):
    url, spec = t
    _save_notebook_from_git(url, spec, LAKEHOUSE_IDS, WAREHOUSE_IDS, WORKSPACE_ID)

if MAX_WORKERS == 1:
    for t in NOTEBOOKS_WITH_DEFAULT_LAKEHOUSES:
        _do_import(t)
else:
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
        list(ex.map(_do_import, NOTEBOOKS_WITH_DEFAULT_LAKEHOUSES))

print("Provisioning complete")


## Done

Next steps for attendees:

* Open any `module_*` notebook in the workspace, they are ready and bound to `silver`.
* Open the **silver** lakehouse: `customers`, `orders`, and the raw files are waiting.
* The **bronze** lakehouse is empty, to be filled during the workshop.
* The **gold** warehouse is empty, populated by the Module 3 interoperability exercise.
